In [11]:
league_id = 73
season_key = '2526'
raw_data_path = r"c:\Users\xrosinach\Desktop\TFM\data\raw"
clean_data_path = r"c:\Users\xrosinach\Desktop\TFM\data\clean"

In [12]:
import os
import pandas as pd
import numpy as np
import json as jsonlib
from typing import Tuple
import unicodedata
import re
from rapidfuzz import process, fuzz

# Obtenemos el CSV con competiciones
cdir = os.getcwd()
utils = os.path.join(os.path.abspath(os.path.join(cdir, '..')), 'utils')
comps = pd.read_csv(os.path.join(utils, 'comps.csv'), sep=';', encoding='latin1')

# JSON con temporadas deseadas
with open(os.path.join(utils, 'des_seasons.json'), 'r', encoding='utf-8') as f:
    desired_seasons = jsonlib.load(f)
act_season = desired_seasons[0]

# Creación de slug a partir de un string.
def create_slug(text: str) -> str:

    text = text.lower()                                                                                     # Letra minúscula
    text = ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')        # Eliminación de acentos
    text = re.sub(r"\s+", "_", text)                                                                        # Substitución de espacios por '_'
    text = re.sub(r"[^a-z0-9_]", "", text)                                                                  # Eliminación de carácteres no alfanuméricos
    text = re.sub(r"_+", "_", text).strip("_")
    
    return text

In [ ]:
# Lectura de los datos de Fotmob
def read_fotmob_data(fotmob_clean_path: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    if not os.path.exists(fotmob_clean_path):
        return None, None, None, None, None, None, None

    standings_path = os.path.join(fotmob_clean_path, 'standings')       # Carpeta con tablas de clasificación
    info_path = os.path.join(fotmob_clean_path, 'info.csv')             # Tabla de información
    matches_path = os.path.join(fotmob_clean_path, 'matches.csv')       # Tabla con partidos jugados

    st_all_path = os.path.join(standings_path, 'all.csv')               # Tablas con las distintas tablas de classificación
    st_home_path = os.path.join(standings_path, 'home.csv')
    st_away_path = os.path.join(standings_path, 'away.csv')
    st_form_path = os.path.join(standings_path, 'form.csv')
    st_xg_path = os.path.join(standings_path, 'xg.csv')

    info_df = pd.read_csv(info_path, sep=';') if os.path.exists(info_path) else None               # Lectura de todos los dataframes si existen
    matches_df = pd.read_csv(matches_path, sep=';') if os.path.exists(matches_path) else None
    all_st_df = pd.read_csv(st_all_path, sep=';') if os.path.exists(st_all_path) else None
    home_st_df = pd.read_csv(st_home_path, sep=';') if os.path.exists(st_home_path) else None
    away_st_df = pd.read_csv(st_away_path, sep=';') if os.path.exists(st_away_path) else None
    form_st_df = pd.read_csv(st_form_path, sep=';') if os.path.exists(st_form_path) else None
    xg_st_df = pd.read_csv(st_xg_path, sep=';') if os.path.exists(st_xg_path) else None
    
    return info_df, matches_df, all_st_df, home_st_df, away_st_df, form_st_df, xg_st_df

# A partir de los datos de Fotmob, obtenemos los nombres de los equipos
def obtain_fotmob_teams(matches_df: pd.DataFrame, all_st_df: pd.DataFrame, home_st_df: pd.DataFrame, away_st_df: pd.DataFrame, form_st_df: pd.DataFrame, xg_st_df: pd.DataFrame) -> list:

    teams_list = []

    if matches_df is not None:                                  # Equipos local y visitante en los partidos
        teams_list.extend(matches_df['home_team'].tolist())
        teams_list.extend(matches_df['away_team'].tolist())

    if all_st_df is not None:                                   # Distintos equipos en la tabla de clasificación
        teams_list.extend(all_st_df['name'].tolist())
    if home_st_df is not None:
        teams_list.extend(home_st_df['name'].tolist())
    if away_st_df is not None:
        teams_list.extend(away_st_df['name'].tolist())
    if form_st_df is not None:
        teams_list.extend(form_st_df['name'].tolist())
    if xg_st_df is not None:
        teams_list.extend(xg_st_df['name'].tolist())

    return sorted(set(teams_list))              # Sin duplicados y ordenado

# Lectura de los datos de Scoresway
def read_scoresway_data(scoresway_clean_path: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not os.path.exists(scoresway_clean_path):
        return None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None

    info_path = os.path.join(scoresway_clean_path, 'info')                # Carpeta con información
    matches_dir_path = os.path.join(scoresway_clean_path, 'matches')          # Tabla con datos de partidos
    standings_path = os.path.join(info_path, 'standings')      # Tablas de clasificación

    managers_path = os.path.join(info_path, 'managers.csv')               # Dataframe con información variada
    matches_path = os.path.join(info_path, 'matches.csv')
    players_path = os.path.join(info_path, 'players.csv')
    teams_path = os.path.join(info_path, 'teams.csv')

    st_total_path = os.path.join(standings_path, 'total.csv')            # Distintas tablas de clasificación
    st_home_path = os.path.join(standings_path, 'home.csv')
    st_away_path = os.path.join(standings_path, 'away.csv')
    st_httotal_path = os.path.join(standings_path, 'half-time-total.csv')
    st_hthome_path = os.path.join(standings_path, 'half-time-home.csv')
    st_htaway_path = os.path.join(standings_path, 'half-time-away.csv')
    st_formhome_path = os.path.join(standings_path, 'form-home.csv')
    st_formaway_path = os.path.join(standings_path, 'form-away.csv')
    st_overunder_path = os.path.join(standings_path, 'over-under.csv')
    st_attendance_path = os.path.join(standings_path, 'attendance.csv')

    matches_info_path = os.path.join(matches_dir_path, 'info.csv')      # Información y estadísticas sobre partidos
    matches_player_stats_path = os.path.join(matches_dir_path, 'player_stats.csv')
    matches_referees_path = os.path.join(matches_dir_path, 'referees.csv')
    matches_team_stats_path = os.path.join(matches_dir_path, 'team_stats.csv')

    managers_df = pd.read_csv(managers_path, sep=';') if os.path.exists(managers_path) else None        # Lectura de todos los CSVs si no existen
    matches_df = pd.read_csv(matches_path, sep=';') if os.path.exists(matches_path) else None
    players_df = pd.read_csv(players_path, sep=';') if os.path.exists(players_path) else None
    teams_df = pd.read_csv(teams_path, sep=';') if os.path.exists(teams_path) else None
    total_st_df = pd.read_csv(st_total_path, sep=';') if os.path.exists(st_total_path) else None
    home_st_df = pd.read_csv(st_home_path, sep=';') if os.path.exists(st_home_path) else None
    away_st_df = pd.read_csv(st_away_path, sep=';') if os.path.exists(st_away_path) else None
    httotal_st_df = pd.read_csv(st_httotal_path, sep=';') if os.path.exists(st_httotal_path) else None
    hthome_st_df = pd.read_csv(st_hthome_path, sep=';') if os.path.exists(st_hthome_path) else None
    htaway_st_df = pd.read_csv(st_htaway_path, sep=';') if os.path.exists(st_htaway_path) else None
    formhome_st_df = pd.read_csv(st_formhome_path, sep=';') if os.path.exists(st_formhome_path) else None
    formaway_st_df = pd.read_csv(st_formaway_path, sep=';') if os.path.exists(st_formaway_path) else None
    overunder_st_df = pd.read_csv(st_overunder_path, sep=';') if os.path.exists(st_overunder_path) else None
    attendance_st_df = pd.read_csv(st_attendance_path, sep=';') if os.path.exists(st_attendance_path) else None
    matches_info_df = pd.read_csv(matches_info_path, sep=';') if os.path.exists(matches_info_path) else None
    matches_player_stats_df = pd.read_csv(matches_player_stats_path, sep=';') if os.path.exists(matches_player_stats_path) else None
    matches_referees_df = pd.read_csv(matches_referees_path, sep=';') if os.path.exists(matches_referees_path) else None
    matches_team_stats_df = pd.read_csv(matches_team_stats_path, sep=';') if os.path.exists(matches_team_stats_path) else None

    return managers_df, matches_df, players_df, teams_df, total_st_df, home_st_df, away_st_df, httotal_st_df, hthome_st_df, htaway_st_df, formhome_st_df, formaway_st_df, overunder_st_df, attendance_st_df, matches_info_df, matches_player_stats_df, matches_team_stats_df, matches_referees_df

# Lectura de los datos de Sofascore
def read_sofascore_data(sofascore_clean_path: str) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if not os.path.exists(sofascore_clean_path):
        return None, None, None, None, None, None, None, None, None, None

    info_path = os.path.join(sofascore_clean_path, 'info')                      # Carpeta con información
    matches_dir_path = os.path.join(sofascore_clean_path, 'matches')            # Tabla con datos de partidos
    standings_path = os.path.join(info_path, 'standings')                       # Tablas de clasificación

    managers_path = os.path.join(info_path, 'managers.csv')               # Dataframe con información variada
    venues_path = os.path.join(info_path, 'venues.csv')
    players_path = os.path.join(info_path, 'players.csv')
    teams_path = os.path.join(info_path, 'teams.csv')

    st_total_path = os.path.join(standings_path, 'total.csv')            # Distintas tablas de clasificación
    st_home_path = os.path.join(standings_path, 'home.csv')
    st_away_path = os.path.join(standings_path, 'away.csv')

    matches_info_path = os.path.join(matches_dir_path, 'matches.csv')      # Información y estadísticas sobre partidos
    matches_lineups_path = os.path.join(matches_dir_path, 'lineups.csv')
    matches_team_stats_path = os.path.join(matches_dir_path, 'statistics.csv')

    managers_df = pd.read_csv(managers_path, sep=';') if os.path.exists(managers_path) else None        # Lectura de todos los CSVs si no existen
    players_df = pd.read_csv(players_path, sep=';') if os.path.exists(players_path) else None
    teams_df = pd.read_csv(teams_path, sep=';') if os.path.exists(teams_path) else None
    venues_df = pd.read_csv(venues_path, sep=';') if os.path.exists(venues_path) else None
    total_st_df = pd.read_csv(st_total_path, sep=';') if os.path.exists(st_total_path) else None
    home_st_df = pd.read_csv(st_home_path, sep=';') if os.path.exists(st_home_path) else None
    away_st_df = pd.read_csv(st_away_path, sep=';') if os.path.exists(st_away_path) else None
    matches_info_df = pd.read_csv(matches_info_path, sep=';') if os.path.exists(matches_info_path) else None
    matches_lineups_df = pd.read_csv(matches_lineups_path, sep=';') if os.path.exists(matches_lineups_path) else None
    matches_statistics_df = pd.read_csv(matches_team_stats_path, sep=';') if os.path.exists(matches_team_stats_path) else None

    return managers_df, players_df, teams_df, venues_df, total_st_df, home_st_df, away_st_df, matches_info_df, matches_lineups_df, matches_statistics_df

# A partir de las listas de las fuentes de los equipos, obtenemos un dataframe con los equipos relacionados
def match_teams(fm_list: list, sw_list: list, ss_list: list, threshold: int = 30) -> pd.DataFrame:

    rows = []
    for team in fm_list:
        match2, score2, _ = process.extractOne(team, sw_list, scorer=fuzz.token_sort_ratio) if len(sw_list) > 0 else ('', 0, '')
        match3, score3, _ = process.extractOne(team, ss_list, scorer=fuzz.token_sort_ratio) if len(ss_list) > 0 else ('', 0, '')
        rows.append({"fotmob": team, "scoresway": match2 if score2 >= threshold else None, "sofascore": match3 if score3 >= threshold else None})

    df = pd.DataFrame(rows)
    df.insert(0, 'team', df['fotmob'].combine_first(df['sofascore']).combine_first(df['scoresway']))        # Columna con el nombre que le vamos a poner al equipo - por prioridad
    return df

# A partir de las listas de las fuentes de los jugadores, obtenemos un dataframe con los jugadores
def match_players(sw_list: list, ss_list: list, threshold: int = 10) -> pd.DataFrame:

    sw_list = sw_list if sw_list is not None else []
    ss_list = ss_list if ss_list is not None else []

    if len(ss_list) == 0 and len(sw_list) == 0:
        return pd.DataFrame(columns=['player', 'sofascore', 'scoresway'])

    if len(ss_list) == 0:
        df = pd.DataFrame({'scoresway': sw_list})
        df['sofascore'] = np.nan
        df.insert(0, 'player', df['scoresway'])
        return df

    rows = []
    for player in ss_list:
        match, score, _ = process.extractOne(player, sw_list, scorer=fuzz.token_sort_ratio) if len(sw_list) > 0 else ('', 0, '')
        rows.append({'sofascore': player, 'scoresway': match if score >= threshold else np.nan})

    df = pd.DataFrame(rows)
    df.insert(0, 'player', df['sofascore'].combine_first(df['scoresway']))

    return df

# Unificación de información del partido
def unify_teams_info(matched_teams: pd.DataFrame, sw_teams_df: pd.DataFrame = None, ss_teams_df: pd.DataFrame = None) -> pd.DataFrame:

    sw_to_team = matched_teams.set_index('scoresway')['team'].dropna().to_dict()        # Mapeamos equipos
    ss_to_team = matched_teams.set_index('sofascore')['team'].dropna().to_dict()

    dfs = []
    if sw_teams_df is not None:                 # Para cada dataframe, si no es vacio, no concatenaremos
        sw_teams_df = sw_teams_df.copy()
        sw_teams_df['TeamName'] = sw_teams_df['club_name'].map(sw_to_team)
        sw_teams_df = sw_teams_df.rename(columns={c: f"{c}_sw" for c in sw_teams_df.columns if c != 'TeamName'})
        dfs.append(sw_teams_df)

    if ss_teams_df is not None:
        ss_teams_df = ss_teams_df.copy()
        ss_teams_df['TeamName'] = ss_teams_df['name'].map(ss_to_team)
        ss_teams_df = ss_teams_df.rename(columns={c: f"{c}_ss" for c in ss_teams_df.columns if c != 'TeamName'})
        dfs.append(ss_teams_df)

    if len(dfs) == 0:
        return matched_teams[['team']].rename(columns={'team': 'TeamName'}).drop_duplicates().reset_index(drop=True)        # En caso que no haya ningún dataframe, añadiremos un dataframe con los nombres de los equipos
    if len(dfs) == 1:
        teams_df = dfs[0]
    else:
        teams_df = pd.merge(dfs[0], dfs[1], on='TeamName', how='outer')

    teams_df = (matched_teams[['team']].rename(columns={'team': 'TeamName'}).drop_duplicates().merge(teams_df, how='left', on='TeamName'))      # Dataframe final
    return teams_df

# Limpieza del dataframe unificado de equipos
def clean_unified_teams(df: pd.DataFrame) -> pd.DataFrame:

    list_columns = df.columns
    df_cleaned = pd.DataFrame()     # Dataframe vacío - ahora vamos a añadir columnas

    df_cleaned['Name'] = df['TeamName']                                         # Vamos añadiendo columnas si existen
    df_cleaned['FullName'] = df['name_sw'] if 'name_sw' in list_columns else df['name_ss'] if 'name_ss' in list_columns else np.nan
    df_cleaned['ShortName'] = df['short_name_ss'] if 'short_name_ss' in list_columns else np.nan
    df_cleaned['Code'] = df['code_sw'] if 'code_sw' in list_columns else np.nan
    df_cleaned['Country'] = df['country_ss'] if 'country_ss' in list_columns else np.nan
    df_cleaned['FoundationDate'] = df['foundation_date_ss'] if 'foundation_date_ss' in list_columns else np.nan
    df_cleaned['VenueCodeSS'] = df['venue_ss'] if 'venue_ss' in list_columns else np.nan
    df_cleaned['ManagerCodeSS'] = df['manager_ss'] if 'manager_ss' in list_columns else np.nan
    df_cleaned['PrimaryColour'] = df['primary_colour_ss'] if 'primary_colour_ss' in list_columns else np.nan
    df_cleaned['SecondaryColour'] = df['secondary_colour_ss'] if 'secondary_colour_ss' in list_columns else np.nan
    df_cleaned['TextColour'] = df['text_colour_ss'] if 'text_colour_ss' in list_columns else np.nan

    df_cleaned['IdSS'] = df['team_id_ss'] if 'team_id_ss' in list_columns else np.nan      # IDs de sofascore y de scoresway
    df_cleaned['IdSW'] = df['sw_id_sw'] if 'sw_id_sw' in list_columns else np.nan

    df_cleaned['FoundationDate'] = pd.to_datetime(df_cleaned['FoundationDate'], unit='s', errors='coerce').dt.strftime('%d/%m/%Y')
    df_cleaned['VenueCodeSS'] = pd.to_numeric(df_cleaned['VenueCodeSS'], errors='coerce').astype('Int64')
    df_cleaned['ManagerCodeSS'] = pd.to_numeric(df_cleaned['ManagerCodeSS'], errors='coerce').astype('Int64')

    df_cleaned.insert(0, 'Slug', df_cleaned['Name'].apply(create_slug))         # Añadimos slug como indice

    return df_cleaned

# Información equipos
def create_teams_info_df(matched_teams: pd.DataFrame, sw_teams_df: pd.DataFrame = None, ss_teams_df: pd.DataFrame = None) -> pd.DataFrame:

    raw_unified_teams_df = unify_teams_info(matched_teams=matched_teams, sw_teams_df=sw_teams_df, ss_teams_df=ss_teams_df)          # Unificación de los equipos en un dataframe conjunto
    teams_df = clean_unified_teams(df=raw_unified_teams_df)                                                                         # Limpieza del dataframe
    return teams_df

# Unificamos información de los jugadores de un único equipo
def unify_players_info(team: str, matched_players: pd.DataFrame, ss_df: pd.DataFrame = None, sw_df: pd.DataFrame = None) -> pd.DataFrame:

    sw_to_player = matched_players.set_index('scoresway')['player'].dropna().to_dict()
    ss_to_player = matched_players.set_index('sofascore')['player'].dropna().to_dict()

    dfs = []

    if ss_df is not None and not ss_df.empty:
        ss_df = ss_df.copy()
        ss_df['PlayerName'] = ss_df['playerName_ss'].map(ss_to_player)
        dfs.append(ss_df)

    if sw_df is not None and not sw_df.empty:
        sw_df = sw_df.copy()
        sw_df['PlayerName'] = sw_df['match_name_sw'].map(sw_to_player)
        dfs.append(sw_df)

    if len(dfs) == 0:
        players_df = (matched_players[['player']].rename(columns={'player':'PlayerName'}).drop_duplicates().reset_index(drop=True))
    elif len(dfs) == 1:
        players_df = dfs[0]
    else:
        players_df = pd.merge(dfs[0], dfs[1], on='PlayerName', how='outer')

    players_df = (matched_players[['player']].rename(columns={'player':'PlayerName'}).drop_duplicates().merge(players_df, how='left', on='PlayerName'))
    players_df.insert(0, 'Team', create_slug(text=team))

    return players_df

# Limpieza del dataframe unificado de jugadores
def clean_unified_players(df: pd.DataFrame) -> pd.DataFrame:

    list_columns = df.columns
    df_cleaned = pd.DataFrame()     # Dataframe vacío - ahora vamos a añadir columnas

    df_cleaned['Name'] = df['PlayerName']                                         # Vamos añadiendo columnas si existen
    df_cleaned['Team'] = df['Team']
    df_cleaned['ShortName'] = df['shortName_ss'] if 'shortName_ss' in list_columns else df['match_name_sw'] if 'match_name_sw' in list_columns else np.nan
    df_cleaned['FirstName'] = df['first_name_sw'] if 'first_name_sw' in list_columns else np.nan
    df_cleaned['SecondName'] = df['last_name_sw'] if 'last_name_sw' in list_columns else np.nan
    df_cleaned['ShortFirstName'] = df['short_first_name_sw'] if 'short_first_name_sw' in list_columns else np.nan
    df_cleaned['ShortSecondName'] = df['short_last_name_sw'] if 'short_last_name_sw' in list_columns else np.nan
    df_cleaned['Country'] = df['country_ss'] if 'country_ss' in list_columns else df['nationality_sw'] if 'nationality_sw' in list_columns else np.nan
    df_cleaned['ShirtNumber'] = df['shirt_num_ss'] if 'shirt_num_ss' in list_columns else df['shirt_number_sw'] if 'shirt_number_sw' in list_columns else np.nan
    df_cleaned['PrefFoot'] = df['pref_foot_ss'] if 'pref_foot_ss' in list_columns else np.nan
    df_cleaned['Height'] = df['height_ss'] if 'height_ss' in list_columns else np.nan
    df_cleaned['DateBirth'] = df['date_birth_ss'] if 'date_birth_ss' in list_columns else np.nan
    df_cleaned['FirstPosition'] = df['first_position_ss'] if 'first_position_ss' in list_columns else np.nan
    df_cleaned['SecondPosition'] = df['second_position_ss'] if 'second_position_ss' in list_columns else np.nan
    df_cleaned['ThirdPosition'] = df['third_position_ss'] if 'third_position_ss' in list_columns else np.nan
    df_cleaned['MarketValue'] = df['market_value_ss'] if 'market_value_ss' in list_columns else np.nan
    df_cleaned['ContractUntil'] = df['contract_until_ss'] if 'contract_until_ss' in list_columns else np.nan

    df_cleaned['IdSS'] = df['playerId_ss'] if 'playerId_ss' in list_columns else np.nan      # IDs de sofascore y de scoresway
    df_cleaned['IdSW'] = df['id_sw'] if 'id_sw' in list_columns else np.nan

    df_cleaned['DateBirth'] = pd.to_datetime(df_cleaned['DateBirth'], unit='s', errors='coerce').dt.strftime('%d/%m/%Y')
    df_cleaned['ContractUntil'] = pd.to_datetime(df_cleaned['ContractUntil'], unit='s', errors='coerce').dt.strftime('%d/%m/%Y')
    df_cleaned['ShirtNumber'] = pd.to_numeric(df_cleaned['ShirtNumber'], errors='coerce').astype('Int64')
    df_cleaned['Height'] = pd.to_numeric(df_cleaned['Height'], errors='coerce').astype('Int64')
    df_cleaned['MarketValue'] = pd.to_numeric(df_cleaned['MarketValue'], errors='coerce').astype('Int64')
    df_cleaned['IdSS'] = pd.to_numeric(df_cleaned['IdSS'], errors='coerce').astype('Int64')

    df_cleaned.insert(0, 'Slug', df_cleaned['Name'].apply(create_slug))         # Añadimos slug como indice

    return df_cleaned.sort_values(by='ShirtNumber')

# Unificación total de los datos de información de jugadores
def create_players_info_df(matched_teams: pd.DataFrame, sw_players_df: pd.DataFrame, ss_players_df: pd.DataFrame) -> pd.DataFrame:

    sw_to_team = matched_teams.set_index('longname_scoresway')['team'].dropna().to_dict()        # Mapeamos equipos
    ss_to_team = matched_teams.set_index('sofascore')['team'].dropna().to_dict()

    if sw_players_df is not None:
        sw_players_df = sw_players_df.copy()
        sw_players_df['TeamName'] = sw_players_df['team'].map(sw_to_team)
        sw_players_df = sw_players_df.rename(columns={c: f"{c}_sw" for c in sw_players_df.columns if c != 'TeamName'})

    if ss_players_df is not None:
        ss_players_df = ss_players_df.copy()
        ss_players_df['TeamName'] = ss_players_df['teamName'].map(ss_to_team)
        ss_players_df = ss_players_df.rename(columns={c: f"{c}_ss" for c in ss_players_df.columns if c != 'TeamName'})

    list_teams = []

    for team in matched_teams['team'].tolist():

        sw_players_df_ = sw_players_df.loc[sw_players_df['TeamName'] == team] if sw_players_df is not None else None
        ss_players_df_ = ss_players_df.loc[ss_players_df['TeamName'] == team] if ss_players_df is not None else None

        if sw_players_df_ is not None and sw_players_df_.empty:
            sw_players_df_ = None
        if ss_players_df_ is not None and ss_players_df_.empty:
            ss_players_df_ = None

        players_names_sw = sw_players_df_['match_name_sw'].dropna().unique().tolist() if sw_players_df_ is not None else []
        players_names_ss = ss_players_df_['playerName_ss'].dropna().unique().tolist() if ss_players_df_ is not None else []

        matched_players = match_players(sw_list=players_names_sw, ss_list=players_names_ss)
        unified_players_df = unify_players_info(team=team, matched_players=matched_players, ss_df=ss_players_df_, sw_df=sw_players_df_)
        cleaned_players_df = clean_unified_players(df=unified_players_df)
        list_teams.append(cleaned_players_df)

    return pd.concat(list_teams, ignore_index=True)

# Unificamos información de los managers
def unify_managers_info(matched_managers: pd.DataFrame, ss_df: pd.DataFrame = None, sw_df: pd.DataFrame = None) -> pd.DataFrame:

    sw_to_player = matched_managers.set_index('scoresway')['player'].dropna().to_dict()
    ss_to_player = matched_managers.set_index('sofascore')['player'].dropna().to_dict()

    dfs = []

    if ss_df is not None and not ss_df.empty:
        ss_df = ss_df.copy()
        ss_df['ManagerName'] = ss_df['name'].map(ss_to_player)
        ss_df = ss_df.rename(columns={c: f"{c}_ss" for c in ss_df.columns if c != 'ManagerName'})
        dfs.append(ss_df)

    if sw_df is not None and not sw_df.empty:
        sw_df = sw_df.copy()
        sw_df['ManagerName'] = sw_df['manager_name'].map(sw_to_player)
        sw_df = sw_df.rename(columns={c: f"{c}_sw" for c in sw_df.columns if c != 'ManagerName'})
        dfs.append(sw_df)

    if len(dfs) == 0:
        managers_df = (matched_managers[['player']].rename(columns={'player':'ManagerName'}).drop_duplicates().reset_index(drop=True))
    elif len(dfs) == 1:
        managers_df = dfs[0]
    else:
        managers_df = pd.merge(dfs[0], dfs[1], on='ManagerName', how='outer')

    managers_df = (matched_managers[['player']].rename(columns={'player':'ManagerName'}).drop_duplicates().merge(managers_df, how='left', on='ManagerName'))

    if sw_df is not None:
        return pd.concat([managers_df, sw_df], ignore_index=True)
    return managers_df

# Limpieza del dataframe unificado de entrenadores
def clean_unified_managers(df: pd.DataFrame) -> pd.DataFrame:

    list_columns = df.columns
    df_cleaned = pd.DataFrame()     # Dataframe vacío - ahora vamos a añadir columnas

    df_cleaned['Name'] = df['ManagerName'] if 'manager_name_sw' not in list_columns else df['ManagerName'].combine_first(df['manager_name_sw'])                 # Vamos añadiendo columnas si existen
    df_cleaned['ShortName'] = df['short_name_ss'] if 'short_name_ss' in list_columns else df['match_name_sw'] if 'match_name_sw' in list_columns else np.nan
    df_cleaned['FirstName'] = df['first_name_sw'] if 'first_name_sw' in list_columns else np.nan
    df_cleaned['SecondName'] = df['last_name_sw'] if 'last_name_sw' in list_columns else np.nan
    df_cleaned['ShortFirstName'] = df['short_first_name_sw'] if 'short_first_name_sw' in list_columns else np.nan
    df_cleaned['ShortSecondName'] = df['short_last_name_sw'] if 'short_last_name_sw' in list_columns else np.nan
    df_cleaned['Country'] = df['country_ss'] if 'country_ss' in list_columns else df['nationality_sw'] if 'nationality_sw' in list_columns else np.nan
    df_cleaned['DateBirth'] = df['date_birth_ss'] if 'date_birth_ss' in list_columns else np.nan
    df_cleaned['Position'] = df['type_sw'] if 'type_sw' in list_columns else np.nan
    df_cleaned['Matches'] = df['matches_ss'] if 'matches_ss' in list_columns else np.nan
    df_cleaned['Wins'] = df['wins_ss'] if 'wins_ss' in list_columns else np.nan
    df_cleaned['Draws'] = df['draws_ss'] if 'draws_ss' in list_columns else np.nan
    df_cleaned['Losses'] = df['losses_ss'] if 'losses_ss' in list_columns else np.nan
    df_cleaned['GoalsFor'] = df['goals_scored_ss'] if 'goals_scored_ss' in list_columns else np.nan
    df_cleaned['GoalsAgainst'] = df['goals_conceded_ss'] if 'goals_conceded_ss' in list_columns else np.nan
    df_cleaned['Points'] = df['points_ss'] if 'points_ss' in list_columns else np.nan

    df_cleaned['IdSS'] = df['id_ss'] if 'id_ss' in list_columns else np.nan      # IDs de sofascore y de scoresway
    df_cleaned['IdSW'] = df['id_sw'] if 'id_sw' in list_columns else np.nan

    df_cleaned['DateBirth'] = pd.to_datetime(df_cleaned['DateBirth'], unit='s', errors='coerce').dt.strftime('%d/%m/%Y')
    df_cleaned['Matches'] = pd.to_numeric(df_cleaned['Matches'], errors='coerce').astype('Int64')
    df_cleaned['Wins'] = pd.to_numeric(df_cleaned['Wins'], errors='coerce').astype('Int64')
    df_cleaned['Draws'] = pd.to_numeric(df_cleaned['Draws'], errors='coerce').astype('Int64')
    df_cleaned['Losses'] = pd.to_numeric(df_cleaned['Losses'], errors='coerce').astype('Int64')
    df_cleaned['GoalsFor'] = pd.to_numeric(df_cleaned['GoalsFor'], errors='coerce').astype('Int64')
    df_cleaned['GoalsAgainst'] = pd.to_numeric(df_cleaned['GoalsAgainst'], errors='coerce').astype('Int64')
    df_cleaned['Points'] = pd.to_numeric(df_cleaned['Points'], errors='coerce').astype('Int64')
    df_cleaned['IdSS'] = pd.to_numeric(df_cleaned['IdSS'], errors='coerce').astype('Int64')

    df_cleaned.insert(0, 'Slug', df_cleaned['Name'].apply(create_slug))         # Añadimos slug como indice

    return df_cleaned

# Dataframe con información de jugadores
def create_managers_info_df(teams_df: pd.DataFrame, sw_managers_df: pd.DataFrame, ss_managers_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:

    if sw_managers_df is not None:
        sw_managers_df['manager_name'] = sw_managers_df['short_first_name'] + ' ' + sw_managers_df['short_last_name']
        list_sw = sw_managers_df['manager_name'].tolist()
    else:
        list_sw = []

    if ss_managers_df is not None:
        list_ss = ss_managers_df['name'].tolist()
    else:
        list_ss = []

    matched_managers = match_players(sw_list=list_sw, ss_list=list_ss)            # Usamos la función de jugadores ya que se puede extrapolar
    managers_df = unify_managers_info(matched_managers=matched_managers, sw_df=sw_managers_df, ss_df=ss_managers_df)
    managers_df = clean_unified_managers(df=managers_df)

    managers_name_dict = managers_df.set_index('IdSS')['Slug'].dropna().to_dict()        # Mapeamos nombres de managers a teams dataframe
    teams_df['ManagerCodeSS'] = teams_df['ManagerCodeSS'].map(managers_name_dict)
    teams_df = teams_df.rename(columns={'ManagerCodeSS': 'Manager'})

    return teams_df, managers_df

# Información de estadios
def create_venues_info_df(teams_df: pd.DataFrame, ss_venues_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:

    if ss_venues_df is None:
        return teams_df, None
    
    venues_df = ss_venues_df.copy()
    venues_df.columns = ['IdSS', 'Name', 'Capacity', 'City', 'Latitude', 'Longitude']
    venues_df = venues_df[['Name', 'Capacity', 'City', 'Latitude', 'Longitude', 'IdSS']]
    venues_df.insert(0, 'Slug', venues_df['Name'].apply(create_slug))                       # Añadimos slug como indice

    venues_name_dict = venues_df.set_index('IdSS')['Slug'].dropna().to_dict()        # Mapeamos nombres de venues a teams dataframe
    teams_df['VenueCodeSS'] = teams_df['VenueCodeSS'].map(venues_name_dict)
    teams_df = teams_df.rename(columns={'VenueCodeSS': 'Venue'})

    return teams_df, venues_df

# Limpieza del dataframe unificado de partidos
def clean_unified_matches(df: pd.DataFrame) -> pd.DataFrame:

    list_columns = df.columns
    df_cleaned = pd.DataFrame()     # Dataframe vacío - ahora vamos a añadir columnas

    df_cleaned['Slug'] = df['Slug']                                                   # Vamos añadiendo columnas si existen
    df_cleaned['Round'] = df['round_ss'] if 'round_ss' in list_columns else df['week_sw'] if 'week_sw' in list_columns else np.nan
    df_cleaned['Date_'] = df['date_time_ss'] if 'date_time_ss' in list_columns else np.nan
    df_cleaned['HomeTeam'] = df['HomeTeam_ss']
    df_cleaned['AwayTeam'] = df['AwayTeam_ss']
    df_cleaned['Winner'] = df['winner_ss'] if 'winner_ss' in list_columns else np.nan
    df_cleaned['HomeScore'] = df['home_score_ss'] if 'home_score_ss' in list_columns else np.nan
    df_cleaned['AwayScore'] = df['away_score_ss'] if 'away_score_ss' in list_columns else np.nan
    df_cleaned['HomeScoreHT'] = df['ht_home_score_sw'] if 'ht_home_score_sw' in list_columns else np.nan
    df_cleaned['AwayScoreHT'] = df['ht_away_score_sw'] if 'ht_away_score_sw' in list_columns else np.nan
    df_cleaned['Attendance'] = df['attendance_ss'] if 'attendance_ss' in list_columns else df['attendance_sw'] if 'attendance_sw' in list_columns else np.nan

    df_cleaned['IdSS'] = df['match_id_ss'] if 'match_id_ss' in list_columns else np.nan         # IDs
    df_cleaned['IdSW'] = df['match_id_sw'] if 'match_id_sw' in list_columns else np.nan

    df_cleaned.insert(2, 'Date', pd.to_datetime(df_cleaned['Date_'], unit='s', errors='coerce').dt.strftime('%d/%m/%Y'))
    df_cleaned.insert(3, 'Time', pd.to_datetime(df_cleaned['Date_'], unit='s', errors='coerce').dt.strftime('%H:%M'))
    df_cleaned = df_cleaned.drop(columns=['Date_'])
    df_cleaned['IdSS'] = pd.to_numeric(df_cleaned['IdSS'], errors='coerce').astype('Int64')
    df_cleaned['Round'] = pd.to_numeric(df_cleaned['Round'], errors='coerce').astype('Int64')
    df_cleaned['HomeScore'] = pd.to_numeric(df_cleaned['HomeScore'], errors='coerce').astype('Int64')
    df_cleaned['AwayScore'] = pd.to_numeric(df_cleaned['AwayScore'], errors='coerce').astype('Int64')
    df_cleaned['HomeScoreHT'] = pd.to_numeric(df_cleaned['HomeScoreHT'], errors='coerce').astype('Int64')
    df_cleaned['AwayScoreHT'] = pd.to_numeric(df_cleaned['AwayScoreHT'], errors='coerce').astype('Int64')
    df_cleaned['Attendance'] = pd.to_numeric(df_cleaned['Attendance'], errors='coerce').astype('Int64')
    df_cleaned['Winner'] = np.select([df_cleaned['Winner'] == 1, df_cleaned['Winner'] == 2, df_cleaned['Winner'] == 3], [df_cleaned['HomeTeam'], df_cleaned['AwayTeam'], 'x'], default=np.nan)      # Nombre de equipo que ha ganado o "X" en caso de empate

    return df_cleaned

# Creación del dataframe con información de los partidos
def create_matches_info_df(teams_df: pd.DataFrame, ss_matches_info_df: pd.DataFrame, sw_matches_info_df: pd.DataFrame) -> pd.DataFrame:

    ss = ss_matches_info_df.copy()              # Copia de Sofascore - siempre lo tendremos
    teams_names_dict = teams_df.set_index('IdSS')['Slug'].dropna().to_dict()
    ss['HomeTeam'] = ss['home_team'].map(teams_names_dict)
    ss['AwayTeam'] = ss['away_team'].map(teams_names_dict)
    ss = ss.rename(columns={c: f"{c}_ss" for c in ss.columns})
    ss['Slug'] = ss['HomeTeam_ss'] + '_' + ss['AwayTeam_ss']

    if sw_matches_info_df is not None:
        sw = sw_matches_info_df.copy()
        teams_names_dict = teams_df.set_index('IdSW')['Slug'].dropna().to_dict()
        sw['HomeTeam'] = sw['home_team_id'].map(teams_names_dict)
        sw['AwayTeam'] = sw['away_team_id'].map(teams_names_dict)
        sw = sw.rename(columns={c: f"{c}_sw" for c in sw.columns})
        sw['Slug'] = sw['HomeTeam_sw'] + '_' + sw['AwayTeam_sw']

        unified_matches_df = ss.merge(sw, how='left', on='Slug')        # Merge en caso de que exista

    else:
        unified_matches_df = ss

    return clean_unified_matches(df = unified_matches_df)

# Limpieza del dataframe de clasificación
def clean_standing(df: pd.DataFrame, rank_status: bool = True) -> pd.DataFrame:

    df_cleaned = pd.DataFrame()
    df_cleaned['Team'] = df['Team']

    cols_map = {'Rank':['rank_sw', 'position_ss', 'idx_fm'],
                'Matches':['played_fm', 'matches_ss', 'matchesPlayed_sw'],
                'Wins':['wins_fm', 'wins_ss', 'matchesWon_sw'],
                'Losses':['losses_fm', 'losses_ss', 'matchesLost_sw'],
                'Draws':['draws_fm', 'draws_ss', 'matchesDrawn_sw'],
                'Points':['pts_fm', 'points_ss', 'points_sw'],
                'GoalsFor':['scores_for_ss', 'goalsFor_sw'],
                'GoalsAgainst':['scores_against_ss', 'goalsAgainst_sw']}

    for new_col, possible_cols in cols_map.items():
        existing_cols = [c for c in possible_cols if c in df.columns]

        if len(existing_cols) > 0:
            df_cleaned[new_col] = df[existing_cols].bfill(axis=1).iloc[:, 0]
        else:
            df_cleaned[new_col] = np.nan

    num_cols = ['Matches', 'Wins', 'Losses', 'Draws', 'GoalsFor', 'GoalsAgainst', 'Points']
    for col in num_cols:
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce').astype('Int64')

    df_cleaned['GoalDiff'] = df_cleaned['GoalsFor'] - df_cleaned['GoalsAgainst']
    df_cleaned['Team'] = df_cleaned['Team'].apply(create_slug)

    if rank_status:
        df_cleaned.insert(2, 'Status', df['promotion_ss'])

    return df_cleaned.sort_values(by='Rank')

# Unificación de standings tables
def unified_standings_tables(matched_teams: pd.DataFrame, fm: pd.DataFrame, ss: pd.DataFrame, sw: pd.DataFrame, rank_status: bool = True) -> pd.DataFrame:

    dfs = []

    if fm is not None:          # Datos Fotmob
        fm_dict = matched_teams.set_index('fotmob')['team'].dropna().to_dict()
        fm['Team'] = fm['name'].map(fm_dict)
        fm = fm.rename(columns={c:f'{c}_fm' for c in fm.columns if c != 'Team'})
        dfs.append(fm)

    if ss is not None:          # Datos Sofascore
        ss_dict = matched_teams.set_index('sofascore')['team'].dropna().to_dict()
        ss['Team'] = ss['team'].map(ss_dict)
        ss = ss.rename(columns={c:f'{c}_ss' for c in ss.columns if c != 'Team'})
        dfs.append(ss)

    if sw is not None:          # Datos Scoresway
        sw_dict = matched_teams.set_index('scoresway')['team'].dropna().to_dict()
        sw['Team'] = sw['contestantClubName'].map(sw_dict)
        sw = sw.rename(columns={c:f'{c}_sw' for c in sw.columns if c != 'Team'})
        dfs.append(sw)

    if len(dfs) == 0:           # Unificamos según lo que tenemos
        unified_standing = pd.DataFrame(columns=['Team'])
    elif len(dfs) == 1:
        unified_standing = dfs[0]
    else:
        unified_standing = dfs[0]
        for df_ in dfs[1:]:
            unified_standing = unified_standing.merge(df_, on='Team', how='outer')

    return clean_standing(unified_standing, rank_status=rank_status)

# Creación de la tabla de clasificación esperada
def create_expected_standing_table(matched_teams: pd.DataFrame, df: pd.DataFrame) -> pd.DataFrame:

    df_clean = df.copy()        # Copia y sacamos columnas
    df_clean = df_clean.drop(columns=['shortName', 'id', 'pageUrl', 'ongoing', 'played', 'wins', 'draws', 'losses', 'scoresStr', 'goalConDiff', 'pts', 'qualColor', 'teamId', 'teamName', 'idx'])

    fm_dict = matched_teams.set_index('fotmob')['team'].dropna().to_dict()      # Diccionario y añadimos nombre
    df_clean['Team'] = df_clean['name'].map(fm_dict)

    df_clean.columns = ['name', 'ExpectedGoalsFor', 'ExpectedGoalsAgainst', 'ExpectedPoints', 'Rank', 'ExpectedGoalsForDiff', 'ExpectedGoalsAgainstDiff', 'ExpectedPointsDiff', 'ExpectedRank', 'ExpectedRankDiff', 'Team']
    df_clean = df_clean[['Team', 'Rank', 'ExpectedRank', 'ExpectedRankDiff', 'ExpectedPoints', 'ExpectedPointsDiff', 'ExpectedGoalsFor', 'ExpectedGoalsAgainst', 'ExpectedGoalsAgainstDiff', 'ExpectedPointsDiff']]
    df_clean['Team'] = df_clean['Team'].apply(create_slug)

    return df_clean.sort_values(by='ExpectedRank')

In [ ]:
league_name = comps[comps['id'] == league_id]['tournament'].iloc[0]     # Nombre de la liga
league_slug = create_slug(text=league_name)                             # Slug de la liga

images_path = os.path.join(raw_data_path, 'images')                             # Imagenes
fotmob_clean_path = os.path.join(clean_data_path, 'fotmob', league_slug, season_key)         # Fotmob
scoresway_clean_path = os.path.join(clean_data_path, 'scoresway', league_slug, season_key)   # Scoresway
sofascore_clean_path = os.path.join(clean_data_path, 'sofascore', league_slug, season_key)   # Sofascore

fm_info_df, fm_matches_df, fm_all_st_df, fm_home_st_df, fm_away_st_df, fm_form_st_df, fm_xg_st_df = read_fotmob_data(fotmob_clean_path=fotmob_clean_path)                                       # Lectura de los datos de Fotmob
fotmob_teams = obtain_fotmob_teams(matches_df=fm_matches_df, all_st_df=fm_all_st_df, home_st_df=fm_home_st_df, away_st_df=fm_away_st_df, form_st_df=fm_form_st_df, xg_st_df=fm_xg_st_df)        # Obtención de los equipos

sw_managers_df, sw_matches_df, sw_players_df, sw_teams_df, sw_total_st_df, sw_home_st_df, sw_away_st_df, sw_httotal_st_df, sw_hthome_st_df, sw_htaway_st_df, sw_formhome_st_df, sw_formaway_st_df, sw_overunder_st_df, sw_attendance_st_df, sw_matches_info_df, sw_matches_player_stats_df, sw_matches_team_stats_df, sw_matches_referees_df = read_scoresway_data(scoresway_clean_path=scoresway_clean_path)     # Lectura de datos de Scoresway
scoresway_teams = sorted(sw_teams_df['club_name'].unique().tolist()) if sw_teams_df is not None else []           # Equipos en scoresway

ss_managers_df, ss_players_df, ss_teams_df, ss_venues_df, ss_total_st_df, ss_home_st_df, ss_away_st_df, ss_matches_info_df, ss_matches_lineups_df, ss_matches_statistics_df = read_sofascore_data(sofascore_clean_path=sofascore_clean_path)        # Lectura de los datos de Sofascore
sofascore_teams = sorted(ss_teams_df['name'].unique().tolist()) if ss_teams_df is not None else []                 # Equipos en sofascore

matched_teams = match_teams(fm_list=fotmob_teams, sw_list=scoresway_teams, ss_list=sofascore_teams)     # Mapeamos equipos
if sw_teams_df is not None:
    sw_long_name_dict = sw_teams_df.set_index('club_name')['name'].dropna().to_dict()                       # Diccionario con los nombres largos en scoresway
    matched_teams['longname_scoresway'] = matched_teams['scoresway'].map(sw_long_name_dict)                 # Aplicamos el diccionario
else:
    matched_teams['longname_scoresway'] = np.nan

# A partir de aquí, con toda la información que tenemos, vamos a crear dataframes unificados a partir de, también, el dataframe de mapeo de equipos
teams_df = create_teams_info_df(matched_teams=matched_teams, sw_teams_df=sw_teams_df, ss_teams_df=ss_teams_df)                          # Información de equipos
players_df = create_players_info_df(matched_teams=matched_teams, sw_players_df=sw_players_df, ss_players_df=ss_players_df)              # Información de jugadores
teams_df, managers_df = create_managers_info_df(teams_df=teams_df, sw_managers_df=sw_managers_df, ss_managers_df=ss_managers_df)        # Información de managers - añadimos manager al dataframe de equipo
teams_df, venues_df = create_venues_info_df(teams_df=teams_df, ss_venues_df=ss_venues_df)                                               # Información de estadios - añadimos estadio al dataframe de equipo
matches_df = create_matches_info_df(teams_df=teams_df, ss_matches_info_df=ss_matches_info_df, sw_matches_info_df=sw_matches_info_df)    # Información de partidos

# Tablas de clasificación
all_standings = unified_standings_tables(matched_teams=matched_teams, fm=fm_all_st_df, ss=ss_total_st_df, sw=sw_total_st_df)
home_standings = unified_standings_tables(matched_teams=matched_teams, fm=fm_home_st_df, ss=ss_home_st_df, sw=sw_home_st_df, rank_status=False)
away_standings = unified_standings_tables(matched_teams=matched_teams, fm=fm_away_st_df, ss=ss_away_st_df, sw=sw_away_st_df, rank_status=False)
half_time_standings = unified_standings_tables(matched_teams=matched_teams, fm=None, ss=None, sw=sw_httotal_st_df, rank_status=False)
expected_standings = create_expected_standing_table(matched_teams=matched_teams, df=fm_xg_st_df)

# CREACIÓ DE DATAFRAMES DE STATS DE JUGADOR I DE EQUIP PER PARTIT
# A PARTIR DE L'ANTERIOR, CREAR UN  DE TEMPORADA

In [47]:
expected_standings

,Team,Rank,ExpectedRank,ExpectedRankDiff,ExpectedPoints,ExpectedPointsDiff,ExpectedGoalsFor,ExpectedGoalsAgainst,ExpectedGoalsAgainstDiff,ExpectedPointsDiff
0,real_madrid,1,1,1,57.184587,5.815413,60.5473,29.969000,-6.969000,5.815413
1,barcelona,2,2,-1,56.603071,10.396929,68.1064,34.218200,-8.218200,10.396929
2,atletico_madrid,3,3,0,45.539380,8.460620,42.4603,29.318300,-4.318300,8.460620
3,real_betis,4,4,1,44.154453,-1.154453,41.7377,32.166200,1.833800,-1.154453
4,athletic_club,5,5,5,43.875976,-8.875976,38.4669,29.114400,7.885600,-8.875976
5,villarreal,6,6,-2,42.817940,11.182060,41.7795,34.438600,-2.438600,11.182060
6,celta_vigo,7,7,-1,41.286464,-1.286464,37.2199,32.262300,-2.262300,-1.286464
7,valencia,8,8,4,39.004480,-7.004480,34.2888,34.640201,6.359799,-7.004480
8,rayo_vallecano,9,9,4,38.567689,-7.567689,36.3799,34.549400,-1.549400,-7.567689
9,real_sociedad,10,10,-2,36.901533,-1.901533,37.4086,39.394400,1.605600,-1.901533


In [31]:
df = fm_xg_st_df.copy()